# SageMaker Endpoint 생성 튜토리얼

이 노트북은 Amazon SageMaker를 사용하여 대규모 언어 모델(LLM) 엔드포인트를 생성하는 방법을 단계별로 안내합니다.

## 학습 목표
- SageMaker 환경 설정 및 초기화
- LMI(Large Model Inference) 컨테이너 이해
- vLLM 설정 구성
- 모델 배포 및 엔드포인트 생성
- 리소스 정리

## 사전 요구사항
- AWS 계정 및 SageMaker 접근 권한
- 충분한 GPU 인스턴스 할당량 (ml.g5.xlarge 이상)
- Hugging Face 토큰 (선택사항, 비공개 모델 사용 시)

## 1. 환경 설정 및 초기화

먼저 필요한 라이브러리를 가져오고 SageMaker 환경을 초기화합니다.

In [ ]:
# 필요한 라이브러리 가져오기
import json
import boto3
import sagemaker
from sagemaker import Model, image_uris, serializers, deserializers

# SageMaker 환경 초기화
# execution role: SageMaker가 다른 AWS 서비스에 접근할 때 사용하는 IAM 역할
role = sagemaker.get_execution_role()

# SageMaker 세션: AWS API와 상호작용하기 위한 세션 객체
sess = sagemaker.session.Session()

# 기본 S3 버킷: 모델 아티팩트와 기타 파일을 저장할 S3 버킷
bucket = sess.default_bucket()

# 현재 리전과 계정 ID 가져오기
region = sess._region_name
account_id = sess.account_id()

# AWS 클라이언트 초기화
# SageMaker 클라이언트: 모델, 엔드포인트 등을 관리
sm_client = boto3.client("sagemaker")
# SageMaker Runtime 클라이언트: 엔드포인트 추론 요청
smr_client = boto3.client("sagemaker-runtime")

# 환경 정보 출력
print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")
print(f"sagemaker version: {sagemaker.__version__}")

## 2. 컨테이너 및 인스턴스 설정

SageMaker에서 대규모 언어 모델을 실행하기 위해 LMI(Large Model Inference) 컨테이너를 사용합니다.

### LMI 컨테이너란?
- AWS에서 제공하는 대규모 모델 추론 전용 컨테이너
- vLLM, TensorRT-LLM 등 최적화된 추론 엔진 포함
- GPU 메모리 효율성과 처리량 최적화

### 인스턴스 타입 선택 가이드
- **ml.g5.xlarge**: 소규모 모델 (7B 이하), 테스트용
- **ml.g6.48xlarge**: 중대형 모델 (13B-70B), 프로덕션 권장
- **ml.p5.48xlarge**: 초대형 모델 (120B+), 최고 성능

In [ ]:
# 리전 설정 - GPU 인스턴스 할당량이 있는 리전 선택
REGION = "us-east-1"

# LMI 컨테이너 버전 선택
# 최신 버전은 다음 링크에서 확인: 
# https://github.com/aws/deep-learning-containers/blob/master/available_images.md#large-model-inference-containers
CONTAINER_VERSION = "0.34.0-lmi16.0.0-cu128"

# 컨테이너 URI 구성
# 763104351884는 AWS DLC(Deep Learning Container) 계정 ID
container_uri = f'763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:{CONTAINER_VERSION}'

# 인스턴스 타입 선택
# 모델 크기와 예산에 따라 선택
instance_type = "ml.g5.xlarge"  # 20B 모델에 적합한 소형 인스턴스

# 더 큰 모델이나 높은 성능이 필요한 경우:
# instance_type = "ml.g6.48xlarge"  # 70B 모델까지 지원
# instance_type = "ml.p5.48xlarge"  # 120B+ 모델, 최고 성능

# 리전 일치 여부 확인
if REGION != sess.boto_region_name:
    print(f"⚠️ Warning: Container region ({REGION}) differs from session region ({sess.boto_region_name})")
    print("   This may cause deployment issues. Consider using the same region.")
else:
    print(f"✅ Region validation passed: {REGION}")
    
print(f"📦 Container URI: {container_uri}")
print(f"🖥️ Instance Type: {instance_type}")

# 💡 팁: 처음 사용하는 경우 작은 인스턴스로 시작해서 테스트 후 확장하는 것을 권장합니다.

## 3. 모델 및 vLLM 설정

사용할 모델과 vLLM 추론 엔진의 설정을 구성합니다.

### vLLM이란?
- 대규모 언어 모델을 위한 고성능 추론 엔진
- PagedAttention으로 메모리 효율성 극대화
- 동적 배치 처리로 처리량 최적화

### 주요 설정 옵션 설명
- **TENSOR_PARALLEL_DEGREE**: GPU 간 모델 분산 정도
- **OPTION_ROLLING_BATCH**: 동적 배치 처리 활성화
- **OPTION_ASYNC_MODE**: 비동기 처리 모드

In [ ]:
# 사용할 모델 지정
# Hugging Face Hub에서 공개된 GPT 모델 사용
HF_MODEL_ID = "openai/gpt-oss-20b"

# Hugging Face 토큰 (선택사항)
# 비공개 모델이나 gated 모델 사용 시 필요
# https://huggingface.co/settings/tokens 에서 생성 가능
HF_TOKEN = ""  # 공개 모델이므로 빈 문자열로 설정

# vLLM 설정 구성
vllm_config = {
    # 기본 모델 설정
    "HF_MODEL_ID": HF_MODEL_ID,           # 사용할 모델 ID
    "HF_TOKEN": HF_TOKEN,                 # Hugging Face 인증 토큰
    
    # 타임아웃 설정
    "OPTION_MODEL_LOADING_TIMEOUT": "1500",  # 모델 로딩 타임아웃 (초) - 큰 모델일수록 더 긴 시간 필요
    
    # 오류 처리 설정
    "SERVING_FAIL_FAST": "true",          # 빠른 실패 모드 활성화 - 문제 발생 시 즉시 오류 반환
    
    # 성능 최적화 설정
    "OPTION_ASYNC_MODE": "true",          # 비동기 처리 모드 활성화 -  동시 요청 처리 성능 향상
    "OPTION_ROLLING_BATCH": "disable",    # 롤링 배치 비활성화

    # GPU 병렬 처리 설정
    "TENSOR_PARALLEL_DEGREE": "max",      # 사용 가능한 모든 GPU 활용 - 단일 GPU 인스턴스에서는 "1"과 동일
    
    # 서비스 엔트리포인트
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service"  # vLLM 비동기 서비스 사용
}

print("✅ vLLM 설정 완료")
print(f"📋 모델: {HF_MODEL_ID}")
print(f"🔧 주요 설정:")
print(f"   - 비동기 모드: {vllm_config['OPTION_ASYNC_MODE']}")
print(f"   - 텐서 병렬도: {vllm_config['TENSOR_PARALLEL_DEGREE']}")
print(f"   - 모델 로딩 타임아웃: {vllm_config['OPTION_MODEL_LOADING_TIMEOUT']}초")

## 4. SageMaker 모델 생성

설정한 컨테이너와 vLLM 구성을 사용하여 SageMaker 모델을 생성합니다.

In [ ]:
# 모델 이름 생성
# HF_MODEL_ID에서 모델명만 추출 (예: "openai/gpt-oss-20b" → "gpt-oss-20b")
HF_MODEL_NAME = HF_MODEL_ID.split('/')[-1]

# SageMaker 모델 객체 생성
model = Model(
    image_uri=container_uri,              # 사용할 컨테이너 이미지
    model_data=None,                      # 모델 데이터 (Hugging Face에서 자동 다운로드)
    role=role,                            # IAM 실행 역할
    env=vllm_config,                      # 환경 변수로 vLLM 설정 전달
    name=sagemaker.utils.name_from_base(HF_MODEL_NAME)  # 고유한 모델 이름 생성
)

print(f"✅ SageMaker 모델 생성 완료")
print(f"📝 모델 이름: {model.name}")
print(f"🐳 컨테이너: {container_uri}")
print(f"🔑 IAM 역할: {role}")

## 5. 엔드포인트 배포

생성한 모델을 실제 추론 엔드포인트로 배포합니다.

### 배포 과정
1. **엔드포인트 구성 생성**: 인스턴스 타입, 개수 등 설정
2. **엔드포인트 생성**: 실제 인프라 프로비저닝
3. **모델 로딩**: 컨테이너에서 모델 다운로드 및 초기화
4. **헬스 체크**: 엔드포인트 정상 동작 확인

⏱️ **예상 소요 시간**: 5-15분 (모델 크기에 따라 다름)

In [ ]:
# 엔드포인트 이름 생성 (타임스탬프 포함으로 고유성 보장)
endpoint_name = sagemaker.utils.name_from_base(HF_MODEL_NAME)

print(f"🚀 엔드포인트 배포 시작...")
print(f"📍 엔드포인트 이름: {endpoint_name}")
print(f"🖥️ 인스턴스: {instance_type} x 1")
print(f"⏱️ 예상 소요 시간: 5-15분")
print("")
print("💡 배포 과정:")
print("   1. 엔드포인트 구성 생성")
print("   2. EC2 인스턴스 프로비저닝")
print("   3. 컨테이너 시작 및 모델 로딩")
print("   4. 헬스 체크 완료")
print("")

# 모델 배포 실행
model.deploy(
    initial_instance_count=1,                           # 초기 인스턴스 개수
    instance_type=instance_type,                        # 인스턴스 타입
    endpoint_name=endpoint_name,                        # 엔드포인트 이름
    container_startup_health_check_timeout=1800,       # 헬스 체크 타임아웃 (30분)
                                                        # 큰 모델은 로딩 시간이 오래 걸림
    wait=False                                          # 비동기 배포 (즉시 반환)
                                                        # True로 설정하면 배포 완료까지 대기
)

print(f"✅ 배포 요청 완료!")
print(f"📊 배포 상태는 다음 셀에서 확인할 수 있습니다.")

## 6. 배포 상태 모니터링

엔드포인트 배포 진행 상황을 AWS 콘솔에서 확인할 수 있습니다.

In [ ]:
from IPython.display import display, HTML

def make_endpoint_link(region, endpoint_name, endpoint_task):
    """AWS 콘솔 엔드포인트 링크 생성 함수"""
    endpoint_link = f'<b><a target="blank" href="https://console.aws.amazon.com/sagemaker/home?region={region}#/endpoints/{endpoint_name}">{endpoint_task} 엔드포인트 상태 확인</a></b>'   
    return endpoint_link 

# AWS 콘솔 링크 생성 및 표시
endpoint_link = make_endpoint_link(region, endpoint_name, '🔗')
display(HTML(endpoint_link))

print("")
print("📊 배포 상태 확인 방법:")
print("   1. 위 링크를 클릭하여 AWS 콘솔로 이동")
print("   2. 엔드포인트 상태 확인:")
print("      - Creating: 생성 중")
print("      - InService: 서비스 준비 완료 ✅")
print("      - Failed: 배포 실패 ❌")
print("   3. CloudWatch 로그에서 상세 정보 확인 가능")
print("")
print("⚠️ 주의사항:")
print("   - 배포 완료까지 5-15분 소요")
print("   - 모델이 클수록 더 오래 걸림")
print("   - 실패 시 CloudWatch 로그 확인 필요")

## 대안: SageMaker JumpStart 사용 (선택사항)

SageMaker JumpStart를 사용하면 사전 구성된 모델을 더 쉽게 배포할 수 있습니다.

### JumpStart의 장점
- 사전 최적화된 모델 구성
- 원클릭 배포
- Inference Components 지원
- 자동 리소스 할당

아래 코드는 JumpStart를 사용한 배포 예시입니다 (현재는 주석 처리됨).

In [ ]:
# SageMaker JumpStart를 사용한 대안적 배포 방법
# 더 큰 모델(120B)이나 Inference Components가 필요한 경우 사용

# from sagemaker.jumpstart.model import JumpStartModel
# from sagemaker.compute_resource_requirements.resource_requirements import ResourceRequirements

# # EULA 동의 (라이선스 조건 확인 후 True로 변경)
# accept_eula = True

# # JumpStart 모델 ID 및 버전
# model_id, model_version = "openai-reasoning-gpt-oss-120b", "1.0.0"

# # 모델 및 엔드포인트 이름 생성
# model_name = endpoint_name = sagemaker.utils.name_from_base("gpt-oss-120b")
# inference_component_name = f"ic-{model_name}"

# # JumpStart 모델 객체 생성
# jumpstart_model = JumpStartModel(
#     model_id=model_id,
#     model_version=model_version,
#     name=model_name
# )

# # Inference Component 기반 배포
# # 여러 모델을 하나의 엔드포인트에서 호스팅 가능
# jumpstart_model.deploy(
#     accept_eula=accept_eula,
#     instance_type="ml.g6.48xlarge",                    # 120B 모델에 적합한 인스턴스
#     initial_instance_count=1,
#     container_startup_health_check_timeout=900,        # 15분 타임아웃
#     endpoint_name=endpoint_name,
#     endpoint_type=sagemaker.enums.EndpointType.INFERENCE_COMPONENT_BASED,
#     inference_component_name=inference_component_name,
#     resources=ResourceRequirements(
#         requests={
#             "num_accelerators": 8,      # GPU 개수
#             "memory": 1024*10,          # 메모리 (MB)
#             "copies": 1,                # 모델 복사본 수
#         }
#     ),
# )

## 7. 엔드포인트 삭제

### 🚨 워크샵 완료 후 반드시 실행하세요! 삭제하지 않으면 계속 비용이 발생합니다!🚨

SageMaker 엔드포인트는 삭제하기 전까지 계속 비용이 발생합니다!

### 💰 예상 비용 (ml.g5.xlarge, us-east-1 리전 기준)
- **1시간**: 약 $1.41
- **하루 (24시간)**: $33.84
- **일주일**: $236.88
- **한 달**: $1,015.20

### 🎯 워크샵 완료 체크리스트
- [ ] 엔드포인트 테스트 완료
- [ ] 학습 내용 정리
- [ ] **엔드포인트 삭제 (필수!)**
- [ ] AWS 콘솔에서 삭제 확인

In [ ]:
# 🚨 워크샵 완료 후 반드시 실행하세요! 🚨\n
# 엔드포인트를 삭제하지 않으면 계속 비용이 발생합니다!\n
import time

try:
    response = sm_client.describe_endpoint(EndpointName=endpoint_name)
    current_status = response['EndpointStatus']
    print(f"현재 엔드포인트 상태: {current_status}")

    if current_status in ['InService', 'Failed']:
        print("\n🗑️ 엔드포인트 삭제를 시작합니다...")

        # 엔드포인트 삭제
        print("   1. 엔드포인트 삭제 중...")
        sess.delete_endpoint(endpoint_name)

        # 엔드포인트 구성 삭제
        print("   2. 엔드포인트 구성 삭제 중...")
        sess.delete_endpoint_config(endpoint_name)

        print("\n✅ 리소스 정리 완료!")
        print("💰 더 이상 비용이 발생하지 않습니다. 워크샵을 안전하게 완료했습니다!")

    elif current_status == 'Creating':
        print("⏳ 엔드포인트가 아직 생성 중입니다.")
        print("   생성 완료 후 다시 이 셀을 실행하여 삭제하세요.")

    else:
        print(f"ℹ️ 현재 상태: {current_status}")

except Exception as e:
    if 'Could not find endpoint' in str(e):
        print("✅ 엔드포인트가 이미 삭제되었거나 존재하지 않습니다.")
        print("💰 비용 발생 없음!")
    else:
        print(f"❌ 오류 발생: {e}")
        print("🔧 수동 삭제 방법:")
        print("   1. AWS 콘솔 → SageMaker → 엔드포인트")
        print(f"   2. '{endpoint_name}' 선택 → 삭제")
        print("   3. 엔드포인트 구성도 함께 삭제")